# Inferencia

Recebe o caminho de uma imagem qualquer e devolve a mascara de instancias colorida
e a contagem. Nao treina nada, so carrega o checkpoint do modelo final da Parte 2
(cabeca de fronteira + watershed).

Se o repo foi clonado do zero, baixa o checkpoint junto ou roda o treino uma vez com
`python scripts/train.py --config configs/dsb2018_boundary.yaml`.

In [ ]:
import sys
sys.path.insert(0, 'src')

import matplotlib.pyplot as plt
import numpy as np

from pa1.inference import load_predictor

## Carrega o modelo

O `Predictor` decide sozinho se roda a imagem inteira ou em tiles. Acima de 768 px de
lado ele passa pro modo tiled com media de logits na sobreposicao, que foi o que a
Parte 4 mostrou funcionar (decodificar por tile e colar quebra os objetos da emenda).

In [ ]:
CONFIG = 'configs/dsb2018_boundary.yaml'
CHECKPOINT = 'runs/dsb2018_boundary/best.pt'

pred = load_predictor(CONFIG, CHECKPOINT)
print('device:', pred.device)
print('checkpoint da epoca', pred.meta.get('epoch'), 'com mAP de validacao', round(pred.meta.get('val_mAP', 0), 4))

## Roda numa imagem

Troca o `IMAGEM` por qualquer caminho. Serve pra imagem de fora do dataset tambem,
so precisa ser microscopia de nucleo pro modelo ter alguma chance.

In [ ]:
from pathlib import Path

# troca por qualquer caminho. o default pega a primeira imagem do dataset so
# pra celula rodar sem editar nada
IMAGEM = sorted(Path('data/dsb2018/stage1_train').glob('*/images/*.png'))[0]

out = pred.predict(IMAGEM)
print('arquivo:', Path(IMAGEM).name[:20])
print('instancias encontradas:', out['count'])
print('rodou em tiles:', out['tiled'])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(out['image']); axes[0].set_title('imagem')
axes[1].imshow(out['colored']); axes[1].set_title(f"instancias ({out['count']})")
axes[2].imshow(out['overlay']); axes[2].set_title('sobreposto')
for ax in axes: ax.axis('off')
plt.tight_layout()

## Os mapas intermediarios

E o que o watershed usa por baixo. `interior` vira marcador, `boundary` e a casca que
separa dois nucleos encostados, e `distance` e o relevo que decide onde cortar.

In [ ]:
maps = out['maps']
fig, axes = plt.subplots(1, len(maps), figsize=(4.5 * len(maps), 4.5))
for ax, (name, m) in zip(np.atleast_1d(axes), maps.items()):
    im = ax.imshow(m, cmap='magma', vmin=0, vmax=1)
    ax.set_title(name); ax.axis('off')
plt.tight_layout()

## Contagem numa pasta inteira

Util pra passar o modelo num lote sem escrever script.

In [ ]:
from pathlib import Path

PASTA = Path('data/dsb2018/stage1_train')
amostras = sorted(PASTA.glob('*/images/*.png'))[:8]

for p in amostras:
    r = pred.predict(p)
    print(f'{p.parent.parent.name[:12]}  {r["count"]:4d} nucleos')